<a href="https://colab.research.google.com/github/Decoding-Data-Science/airesidency/blob/main/Cohort12_groq_sdk-history-assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

3 things we require
1. Library- groq (version 1.6- aug 2026)
2. keys
3. code - app.py

In [1]:
pip install groq

In [2]:
import os
from google.colab import userdata
groq_api_key = userdata.get("GROQ_API_KEY")

In [4]:
import os
from groq import Groq

os.environ["GROQ_API_KEY"] = groq_api_key

client = Groq(
    api_key=groq_api_key,
)

chat_completion = client.chat.completions.create(
    messages=[
        {
            "role": "user",
            "content": "Explain the importance of fast language models",
        }
    ],
    model="openai/gpt-oss-120b",
)

print(chat_completion.choices[0].message.content)

## Why Speed Matters for Modern Language Models  

Large language models (LLMs) have become the backbone of countless products—chatbots, search assistants, code generators, translation services, and more.  While raw capability (the ability to generate fluent, accurate, and creative text) is crucial, **speed** is often the make‑or‑break factor for real‑world success. Below is a structured overview of why fast LLMs are important, the concrete benefits they unlock, and the trade‑offs that engineers and product teams need to manage.

---

### 1. User‑Facing Experience  

| Aspect | Why Speed Helps | Real‑World Example |
|--------|----------------|-------------------|
| **Responsiveness** | Humans expect sub‑second replies for conversational UI; latency > 500 ms feels “slow.” | A messaging‑app chatbot that answers in ~150 ms keeps the conversation flow natural. |
| **Retention & Engagement** | Faster responses increase session length and reduce drop‑off. | Voice assistants (e.g., smart spea

In [5]:
import gradio as gr

def chat_with_groq(message):
    chat_completion = client.chat.completions.create(
        messages=[
            {
        "role": "system",
        "content": "you are python bot , answer only python related question"
      },
            {
                "role": "user",
                "content": message,
            }
        ],
        model="openai/gpt-oss-120b", # Using the same model as before
    )
    return chat_completion.choices[0].message.content

iface = gr.Interface(fn=chat_with_groq, inputs="textbox", outputs="textbox", title="Groq Chat with openai DDS")
iface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2724c37e5f77986afb.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [9]:
import gradio as gr
from groq import Groq
import os

# Initialize Groq client using stored API Key
client = Groq(api_key=os.environ.get("GROQ_API_KEY", groq_api_key))

def chat_with_groq(message, history_state):
    if not message.strip():
        return "", history_state, ""

    # Append user message
    history_state.append({"role": "user", "content": message})

    # call API
    try:
        chat_completion = client.chat.completions.create(
            messages=history_state,
            model="openai/gpt-oss-120b",
        )
        reply = chat_completion.choices[0].message.content
        raw_json = chat_completion.model_dump_json(indent=2)
    except Exception as e:
        reply = f"Error: {str(e)}"
        raw_json = "{}"

    # Append assistant reply to keep memory
    history_state.append({"role": "assistant", "content": reply})

    # Return values: (cleared input, updated state, response markdown, raw json response)
    return "", history_state, reply, raw_json

# Custom styling for a polished, dark/light balanced theme compatible with Gradio 6.0
custom_css = """
.title-container { text-align: center; margin-bottom: 20px; }
.header-badge { background-color: #3b82f6; color: white; padding: 4px 12px; border-radius: 12px; font-size: 0.85em; font-weight: bold; display: inline-block; }
.input-box textarea { font-family: 'Fira Code', monospace; }
"""

with gr.Blocks(title="PyGPT - Professional Python Assistant") as demo:
    # Initialize persistent memory state with system message
    history_state = gr.State([
        {"role": "system", "content": "you are python bot , answer only python related question"}
    ])

    # Use gr.HTML for custom div container and modern header presentation
    gr.HTML("""
    <div class='title-container'>
        <span class='header-badge'>POWERED BY GROQ & LLAMA 3.3</span>
        <h1 style='text-align: center; margin-top: 10px; font-family: sans-serif;'>🐍 PyGPT: Dedicated Python Expert</h1>
        <p style='text-align: center; color: var(--body-text-color-subdued);'>Get instant, production-ready code explanations, debugging tips, and Python solutions.</p>
    </div>
    """)

    with gr.Row():
        # Left Column - Controls & Input
        with gr.Column(scale=2):
            gr.Markdown("### 💬 Ask a Question")
            message_input = gr.Textbox(
                lines=6,
                label="Python Query / Code Snippet",
                placeholder="Explain lambda functions or paste your bug here...",
                elem_classes="input-box"
            )
            with gr.Row():
                submit_btn = gr.Button("🚀 Run Query", variant="primary")
                clear_btn = gr.Button("🧹 Clear Input", variant="secondary")

        # Right Column - Output & API Analytics
        with gr.Column(scale=3):
            gr.Markdown("### 🤖 Response Workspace")
            with gr.Tabs():
                with gr.TabItem("Assistant Reply"):
                    chatbot_output = gr.Markdown(value="*Waiting for your query...*")
                with gr.TabItem("Response Metadata (JSON)"):
                    json_output = gr.Code(language="json", label="Raw API Response")

    # Submit Actions
    submit_btn.click(
        fn=chat_with_groq,
        inputs=[message_input, history_state],
        outputs=[message_input, history_state, chatbot_output, json_output]
    )

    # Reset Action
    def reset_chat():
        return "", [{"role": "system", "content": "you are python bot , answer only python related question"}], "*Chat cleared.*", ""

    clear_btn.click(
        fn=reset_chat,
        inputs=[],
        outputs=[message_input, history_state, chatbot_output, json_output]
    )

# Correctly pass CSS and Theme to launch in Gradio 6.0+
demo.launch(share=True, css=custom_css, theme=gr.themes.Soft())

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5c07b821f34c6870b3.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [6]:
import gradio as gr

def chat_with_groq(message):
    chat_completion = client.chat.completions.create(
        messages=[
            {
                "role": "system",
                "content": "you are python bot , answer only python related question"
            },
            {
                "role": "user",
                "content": message,
            }
        ],
        model="llama-3.3-70b-versatile",
    )
    return chat_completion.choices[0].message.content


with gr.Blocks(theme=gr.themes.Soft(), css="""
    .gradio-container {
        border: 2px solid #4CAF50;
        border-radius: 10px;
        padding: 20px;
        box-shadow: 5px 5px 15px rgba(0,0,0,0.1);
    }
    .gr-textbox {
        border: 1px solid #ccc;
        border-radius: 5px;
    }
    .gradio-column {
        border: 1px dashed #ddd;
        padding: 10px;
        border-radius: 8px;
        margin: 5px;
    }
""") as demo:
    gr.Markdown("# <center>Groq Chat with Llama 3.3 DDS</center>")
    gr.Markdown("Ask any Python-related questions to the Llama 3.3 bot!")

    with gr.Row():
        with gr.Column(scale=2):
            message_input = gr.Textbox(lines=5, label="Your Message", placeholder="Type your Python question here...")
            with gr.Row():
                submit_btn = gr.Button("Send", variant="primary")
                clear_btn = gr.Button("Clear", variant="secondary")
        with gr.Column(scale=3):
            chatbot_output = gr.Textbox(lines=10, label="Bot's Response", interactive=False)

    submit_btn.click(chat_with_groq, inputs=message_input, outputs=chatbot_output)
    clear_btn.click(lambda: "", outputs=message_input, queue=False)
    clear_btn.click(lambda: "", outputs=chatbot_output, queue=False)

demo.launch()

/tmp/ipykernel_3459/2616525125.py:20: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), css="""


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://edfc4b8acebce1d9b6.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import gradio as gr

def chat_with_groq(message):
    chat_completion = client.chat.completions.create(
        messages=[
            {
                "role": "system",
                "content": "you are python bot , answer only python related question"
            },
            {
                "role": "user",
                "content": message,
            }
        ],
        model="llama-3.3-70b-versatile",
    )
    # Return both the message content and the full JSON representation
    return chat_completion.choices[0].message.content, chat_completion.model_dump_json(indent=2)


with gr.Blocks(title="Groq Chat with Llama 3.3 DDS") as demo:
    gr.Markdown("# Groq Chat with Llama 3.3 DDS")
    gr.Markdown("Ask any Python-related questions to the Llama 3.3 bot!")

    message_input = gr.Textbox(lines=5, label="Your Message", placeholder="Type your Python question here...")
    submit_btn = gr.Button("Send")

    with gr.Row():
        chatbot_output = gr.Textbox(lines=10, label="Bot's Response", interactive=False)
        json_output = gr.Textbox(lines=10, label="Full JSON Response", interactive=False)

    submit_btn.click(
        chat_with_groq,
        inputs=message_input,
        outputs=[chatbot_output, json_output]
    )

demo.launch(share=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ba4e400263553aa7e8.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



# Understanding the 3 Roles in Chat API

## Think of it like a restaurant

| Role | Who is it? | Example |
|------|-----------|---------|
| `system` | Manager's instructions (user never sees this) | "Only answer Python topics" |
| `user` | The human asking the question | "What is a list?" |
| `assistant` | The bot's reply (saved for memory) | "A list is a collection..." |

## Why do we save messages in history?

- **Without history** → bot forgets everything (every message = new conversation)
- **With history** → bot remembers the full chat (can handle follow-ups)

## Example flow

```
history = [system]                            → bot knows HOW to behave
history = [system, user1]                     → bot sees 1st question
history = [system, user1, assistant1]         → bot remembers its answer
history = [system, user1, assistant1, user2]  → bot sees follow-up WITH context
```

## One line summary

> `system` = rules, `user` = question, `assistant` = answer.
> Save all three in `history` so the bot remembers the conversation.

In [ ]:
from google.colab import userdata
from groq import Groq

# Get key from Colab secrets
client = Groq(api_key=userdata.get("GROQ_API_KEY"))

# Step 1: System message (manager instruction)
history = [
    {"role": "system", "content": "you are python bot, answer only python related question"}
]

# Step 2: Chat loop
while True:
    user_input = input("You: ")

    if user_input.lower() == "quit":
        print("Bye!")
        break

    # Step 3: Save user message
    history.append({"role": "user", "content": user_input})

    # Step 4: Send to Groq
    response = client.chat.completions.create(
        messages=history,
        model="llama-3.3-70b-versatile",
    )

    reply = response.choices[0].message.content

    # Step 5: Save assistant reply
    history.append({"role": "assistant", "content": reply})

    print(f"Bot: {reply}")

You: hi
Bot: Hello. How can I assist you with Python today? Do you have a coding question or need help with a specific topic?
You: what is best place in dubai
Bot: This conversation is about Python programming. If you have a Python-related question, I'd be happy to help. If you're looking for recommendations about places in Dubai, I can provide a Python script to help you find the best spots.

For example, I can give you a Python script that uses APIs to fetch reviews and ratings of popular places in Dubai. Would you like that? 

Here's a sample script:
```python
import requests

def get_place_reviews(api_key, location):
    url = f"https://maps.googleapis.com/maps/api/place/textsearch/json?query={location}&key={api_key}"
    response = requests.get(url)
    data = response.json()
    return data

# Example usage:
api_key = "YOUR_API_KEY"
location = "Dubai"
reviews = get_place_reviews(api_key, location)
print(reviews)
```
Please note that you need to have a Google Maps API key to use t